This notebook rebuilds document-level LLM source mentions from `llm_labelled_results_master.csv`, applies the manually reviewed source-name decisions, then joins the retained source labels back to `processed_data/merged_all.csv`. It treats `llm_labelled_data/compiled/manually_cleaned/llm_labelled_datasource_review_final.xlsx` as the authoritative review file, writes a conflict-review workbook before collapsing review decisions, and keeps only accepted/relabelled data-resource/product and data-holder rows with `document_count >= 2` and `proposed_standardized_name_document_count > 50`.

All paths are resolved through `project_config.py`. Set `DATA_INTERDEPENDENCIES_DATA_ROOT` before running the notebook.


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

import ast
import json
from pathlib import Path
import re

import numpy as np
import pandas as pd

# Local testing root. Later, replace this single value with the envpol share-drive root.
# DATA_ROOT is configured through project_config.
ROOT = DATA_ROOT
CODE_ROOT = REPO_ROOT
TEXT_MINING_CODE_ROOT = CODE_ROOT / "text_mining"

compiled_path = DATA_ROOT / "llm_labelled_data" / "compiled"
processed_path = DATA_ROOT / "processed_data"
processed_path.mkdir(parents=True, exist_ok=True)

llm_master_file = compiled_path / "llm_labelled_results_master.csv"
manual_review_file = compiled_path / "manually_cleaned" / "llm_labelled_datasource_review_final.xlsx"
source_mentions_long_file = processed_path / "standardized_source_mentions_long.csv"
standardization_audit_file = processed_path / "standardized_source_mentions_audit.csv"
merged_all_file = processed_path / "merged_all.csv"

merged_output_file = processed_path / "llm_labelled_data_sources_merged_bibliometric.csv"
unmatched_output_file = processed_path / "llm_labelled_data_sources_unmatched_bibliometric.csv"
coverage_output_file = processed_path / "llm_labelled_data_sources_merge_coverage.csv"
asjc_subject_fields_output_file = processed_path / "llm_labelled_data_sources_asjc_subject_fields.csv"
asjc_subjects_long_output_file = processed_path / "llm_labelled_data_sources_asjc_subjects_long.csv"
supplementary_workbook_file = processed_path / "interdependence_paper_supplementary_data.xlsx"
review_conflict_workbook_file = processed_path / "standardized_source_metadata_conflict_review.xlsx"

REVIEW_ACTIONS_TO_USE = {"accept", "relabel"}
MIN_REVIEW_DOCUMENT_COUNT = 2
# Exclude proposed standardized-name groups with 50 or fewer documents.
MIN_STANDARDIZED_NAME_DOCUMENT_COUNT_EXCLUSIVE = 50
INCLUDED_ENTITY_TYPES = [
    "data_resource_or_product",
    "data_holder",
]

# Keep this True while manually reviewing conflicts. Set to False only after the conflict workbook is clean.
STOP_FOR_MANUAL_CONFLICT_REVIEW = True

chunk_size = 200_000

print(f"Data root: {DATA_ROOT}")
print(f"LLM master file used to rebuild source mentions: {llm_master_file}")
print(f"Manual review workbook: {manual_review_file}")
print(f"Standardized source mentions output: {source_mentions_long_file}")
print(f"Bibliometric input: {merged_all_file}")
print(f"Supplementary workbook output: {supplementary_workbook_file}")
print(f"Review conflict workbook output: {review_conflict_workbook_file}")


In [ ]:
def normalize_doi(value):
    if pd.isna(value):
        return None

    doi = str(value).strip()
    if not doi or doi.lower() in {"nan", "<na>", "none"}:
        return None

    doi = re.sub(r"^(https?://(dx\.)?doi\.org/)", "", doi, flags=re.IGNORECASE)
    doi = re.sub(r"^doi\s*:\s*", "", doi, flags=re.IGNORECASE)
    doi = doi.strip().strip("\"'[](){}<>")
    doi = re.sub(r"\s+", "", doi)
    doi = doi.rstrip("/")

    return doi.lower() or None


def doi_to_file_name_key(value):
    doi = normalize_doi(value)
    if doi is None:
        return None
    return f"{doi.replace('/', '_')}.txt".casefold()


def normalize_file_name_key(value):
    if pd.isna(value):
        return None

    file_name = str(value).strip()
    if not file_name or file_name.lower() in {"nan", "<na>", "none"}:
        return None

    return Path(file_name).name.casefold()


TEXT_IDENTIFIER_COLUMNS = [
    "global_index",
    "file_name",
    "doi",
    "relative_path",
    "source_subfolder",
]


def clean_text_value(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.lower() in {"nan", "<na>", "none"}:
        return ""
    return text


FINAL_DATA_HOLDER_TYPE_LABELS = {
    "public": "National-public",
    "oublic": "National-public",
    "university": "Research",
    "research": "Research",
    "intergovernmental_or_international": "Transnational",
    "intergovernmental": "Transnational",
    "international": "Transnational",
    "private": "Private",
    "ngo": "NGO",
    "hybrid": "Hybrid",
}
DATA_HOLDER_TYPE_OVERRIDES_BY_HOLDER = {
    # DSMZ is a research institution, so treat it with the research-holder group.
    "Leibniz Institute (DSMZ)": "Research",
}
FINAL_DATA_HOLDER_NAME_LABELS = {
    # Short paper-figure labels for frequent long holder names.
    "DOE Joint Genome Institute": "US DOE",
    "European Environment Agency": "EEA",
    "World Climate Research Programme (WCRP)": "WCRP",
    "Michigan State University": "MSU",
    "University of Wisconsin-Madison": "UW-Madison",
    "University of Wisconsin–Madison": "UW-Madison",
    "University of Colorado Boulder": "CU Boulder",
    "University of East Anglia": "UEA",
    "University of British Columbia": "UBC",
    "University of California, Los Angeles": "UCLA",
    "University of California, Santa Barbara": "UCSB",
    "University of California, Santa Cruz": "UCSC",
}
def split_clean_semicolon_values(value):
    parts = []
    for part in clean_text_value(value).replace("\xa0", " ").split(";"):
        text = re.sub(r"\s+", " ", clean_text_value(part)).strip()
        if text:
            parts.append(text)
    return parts


def final_data_holder_type_label(value):
    # Export final, paper-facing holder-type labels from notebook 6.
    labels = []
    seen = set()
    for part in split_clean_semicolon_values(value):
        normalized = re.sub(r"[^a-z0-9]+", "_", part.casefold()).strip("_")
        label = FINAL_DATA_HOLDER_TYPE_LABELS.get(normalized, part)
        key = label.casefold()
        if label and key not in seen:
            seen.add(key)
            labels.append(label)
    return "; ".join(labels)


def final_data_holder_name_label(value):
    # Keep long institutional names readable in downstream figures.
    text = clean_text_value(value)
    if not text:
        return ""

    if text in FINAL_DATA_HOLDER_NAME_LABELS:
        return FINAL_DATA_HOLDER_NAME_LABELS[text]

    # Prefer reviewed acronyms already present in parentheses, e.g. DSMZ, UCSD, or iDiv.
    # The pattern requires at least one letter, so year notes like (2006) are not treated as acronyms.
    acronym_match = re.search(r"\(([A-Za-z][A-Za-z0-9&.-]{1,15})\)$", text)
    if acronym_match:
        return acronym_match.group(1)

    if text.startswith("University of "):
        return "U. " + text.removeprefix("University of ")
    if text.endswith(" University"):
        return text.removesuffix(" University") + " U."

    return text


def final_data_holder_name_list(value):
    # Apply holder-name labels to each item in a semicolon-separated holder list.
    labels = []
    seen = set()
    for part in split_clean_semicolon_values(value):
        label = final_data_holder_name_label(part)
        key = label.casefold()
        if label and key not in seen:
            seen.add(key)
            labels.append(label)
    return "; ".join(labels)


def apply_final_name_labels(frame):
    # Add exactly one presentation-only holder-name column without changing canonical names.
    updated = frame.copy()

    if "data_holder" in updated.columns:
        updated["data_holder_for_visualization"] = updated["data_holder"].map(final_data_holder_name_list)

    return updated


def apply_final_data_holder_type_labels(frame):
    # Holder-type edits are real corrections/reporting labels, so write them
    # back to the canonical data_holder_type column.
    updated = frame.copy()
    if "data_holder_type" not in updated.columns:
        return updated

    for holder_name, holder_type in DATA_HOLDER_TYPE_OVERRIDES_BY_HOLDER.items():
        holder_mask = pd.Series(False, index=updated.index)
        if "standardized_name" in updated.columns:
            holder_mask = holder_mask | updated["standardized_name"].map(clean_text_value).eq(holder_name)
        if "data_holder" in updated.columns:
            holder_mask = holder_mask | updated["data_holder"].map(
                lambda value: holder_name in split_clean_semicolon_values(value)
            )
        updated.loc[holder_mask, "data_holder_type"] = holder_type

    updated["data_holder_type"] = updated["data_holder_type"].map(final_data_holder_type_label)
    return updated


def apply_final_supplement_labels(frame):
    # One call adds name-only visualization labels and canonical holder-type corrections before export.
    return apply_final_data_holder_type_labels(apply_final_name_labels(frame))


def ordered_unique_join(values):
    seen = []
    for value in values:
        text = clean_text_value(value)
        if text and text not in seen:
            seen.append(text)
    return "; ".join(seen)


def ordered_unique_semicolon_join(values):
    seen = set()
    parts = []

    for value in values:
        for part in clean_text_value(value).replace("\xa0", " ").split(";"):
            text = re.sub(r"\s+", " ", clean_text_value(part)).strip()
            key = text.casefold()
            if text and key not in seen:
                seen.add(key)
                parts.append(text)

    return "; ".join(parts)


def category_column_name(entity_type):
    return f"{entity_type}_names"


def parse_list_field(value):
    text = clean_text_value(value)
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        parsed = text

    if isinstance(parsed, (list, tuple, set)):
        return [str(item).strip() for item in parsed if str(item).strip()]

    return [part.strip() for part in text.split(";") if part.strip()]


def join_list_field(value):
    return ordered_unique_join(parse_list_field(value))


BRACKETED_NUMERIC_SOURCE_RE = re.compile(
    r"^(?:\[\s*-?\d+(?:\s*[,;/\-]\s*-?\d+)*\s*\]\s*)+$"
)
COMMA_SOURCE_LIST_PART_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9 .&'/()\-]{1,80}$")
YEAR_ONLY_PART_RE = re.compile(r"^(?:19|20)\d{2}[a-z]?$", flags=re.IGNORECASE)


def parse_json_like_source_text(text):
    candidates = [text]
    overescaped_quote = "\\\\" + chr(34)
    escaped_quote = "\\" + chr(34)
    if overescaped_quote in text:
        candidates.append(text.replace(overescaped_quote, escaped_quote))

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except (json.JSONDecodeError, TypeError):
            continue
    return None


def numeric_json_list_to_reference_marker(items):
    if not items or not all(isinstance(item, (int, float)) and not isinstance(item, bool) for item in items):
        return None
    labels = []
    for item in items:
        labels.append(str(int(item)) if isinstance(item, float) and item.is_integer() else str(item))
    return "[" + ",".join(labels) + "]"


def split_delimited_source_text(text):
    semi_parts = [part.strip() for part in str(text or "").split(";") if part.strip()]
    if len(semi_parts) != 1:
        return semi_parts

    only_part = semi_parts[0]
    comma_parts = [part.strip() for part in only_part.split(",") if part.strip()]
    if len(comma_parts) < 3:
        return semi_parts
    if any(YEAR_ONLY_PART_RE.match(part) for part in comma_parts):
        return semi_parts
    plausible_parts = [part for part in comma_parts if COMMA_SOURCE_LIST_PART_RE.match(part)]
    short_or_acronym_parts = [
        part for part in comma_parts
        if re.fullmatch(r"[A-Z][A-Z0-9.&/ -]{1,25}", part) or len(part.split()) <= 5
    ]
    if len(plausible_parts) == len(comma_parts) and len(short_or_acronym_parts) >= 2:
        return comma_parts
    return semi_parts


def extract_source_strings(parsed):
    if isinstance(parsed, list):
        numeric_marker = numeric_json_list_to_reference_marker(parsed)
        if numeric_marker:
            return [numeric_marker]

        cleaned_items = []
        for item in parsed:
            cleaned_items.extend(extract_source_strings(item))
        return cleaned_items

    if isinstance(parsed, dict):
        item = parsed.get("name") or parsed.get("data_source") or parsed.get("source")
        return extract_source_strings(item) if item is not None else []

    if isinstance(parsed, str):
        text = parsed.strip()
        if not text:
            return []
        if BRACKETED_NUMERIC_SOURCE_RE.match(text):
            return [text]
        nested = parse_json_like_source_text(text)
        if isinstance(nested, list):
            return extract_source_strings(nested)
        return split_delimited_source_text(text)

    text = str(parsed).strip()
    return [text] if text else []


def parse_data_sources(value):
    text = clean_text_value(value)
    if not text:
        return []

    parsed = parse_json_like_source_text(text)
    if parsed is None:
        parsed = text

    seen = []
    for item in extract_source_strings(parsed):
        item = clean_text_value(item)
        if item and item not in seen:
            seen.append(item)
    return seen


def source_doi_from_file_name(file_name):
    stem = Path(str(file_name or "")).name
    stem = re.sub(r"\.txt$", "", stem, flags=re.IGNORECASE).strip()
    match = re.match(r"^(10\.\d{4,9})[_/](.+)$", stem, flags=re.IGNORECASE)
    if not match:
        return ""
    suffix = match.group(2).replace("_", "/")
    return f"https://doi.org/{match.group(1)}/{suffix}"


def review_lookup_key(value):
    text = clean_text_value(value).casefold()
    return " ".join("".join(char if char.isalnum() else " " for char in text).split())


REVIEW_ACTION_ALIASES = {
    "accepted": "accept",
    "ok": "accept",
    "use": "accept",
    "keep": "accept",
    "re_label": "relabel",
    "relabeled": "relabel",
    "relabelled": "relabel",
}


def normalize_review_action(value):
    text = clean_text_value(value).casefold()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return REVIEW_ACTION_ALIASES.get(text, text)


ENTITY_TYPE_ALIASES = {
    "data_resource_product": "data_resource_or_product",
    "data_resource": "data_resource_or_product",
    "data_product": "data_resource_or_product",
    "resource": "data_resource_or_product",
    "product": "data_resource_or_product",
    "data_holder_or_provider": "data_holder",
    "data_holder_provider": "data_holder",
    "data_provider": "data_holder",
    "holder": "data_holder",
    "provider": "data_holder",
}


def normalize_entity_type(value):
    text = clean_text_value(value).casefold()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return ENTITY_TYPE_ALIASES.get(text, text)


def require_columns(frame, required_columns, label):
    missing_columns = set(required_columns) - set(frame.columns)
    if missing_columns:
        raise ValueError(f"{label} is missing required columns: {sorted(missing_columns)}")


def ensure_column(frame, column, aliases=None, default=""):
    if column in frame.columns:
        return
    aliases = aliases or []
    for alias in aliases:
        if alias in frame.columns:
            frame[column] = frame[alias]
            return
    frame[column] = default


def read_review_workbook(path):
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=0)
    return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)


# URL differences are preserved in downstream metadata but are not treated as manual-review conflicts.
REVIEW_METADATA_CONFLICT_FIELDS = [
    "entity_type",
    "data_holder",
    "data_holder_type",
    "data_holder_country_ISO2",
]
RAW_LOOKUP_CONFLICT_FIELDS = [
    "standardized_name",
    *REVIEW_METADATA_CONFLICT_FIELDS,
]
REVIEW_EXCEL_CELL_CHAR_LIMIT = 32_767
REVIEW_EXCEL_ILLEGAL_CHARACTERS_RE = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")


def distinct_nonempty_values(values):
    distinct_values = []
    for value in values:
        text = clean_text_value(value)
        if text and text not in distinct_values:
            distinct_values.append(text)
    return distinct_values


def joined_distinct_values(values):
    return "; ".join(distinct_nonempty_values(values))


def conflict_fields_for_group(group, fields):
    conflict_fields = []
    for field in fields:
        if field not in group.columns:
            continue
        if len(distinct_nonempty_values(group[field])) > 1:
            conflict_fields.append(field)
    return conflict_fields


def summarize_review_conflicts(retained_review_rows, group_column, conflict_fields):
    # Keep the manual-review workbook compact: first the identifier, then the conflicting values,
    # then only a few context columns at the far right.
    value_columns = [f"{field}_values" for field in conflict_fields]
    context_columns = ["raw_names", "document_count_sum", "review_row_count"]
    output_columns = [group_column, "conflict_fields", *value_columns, *context_columns]

    if retained_review_rows.empty:
        return pd.DataFrame(columns=output_columns)

    summary_rows = []
    for group_value, group in retained_review_rows.groupby(group_column, dropna=False):
        group_text = clean_text_value(group_value)
        if not group_text:
            continue

        conflicting_fields = conflict_fields_for_group(group, conflict_fields)
        if not conflicting_fields:
            continue

        document_counts = pd.to_numeric(group["document_count"], errors="coerce").fillna(0)
        summary_row = {
            group_column: group_text,
            "conflict_fields": "; ".join(conflicting_fields),
            "raw_names": joined_distinct_values(group["raw_name"]),
            "document_count_sum": int(document_counts.sum()),
            "review_row_count": len(group),
        }
        for field in conflict_fields:
            summary_row[f"{field}_values"] = joined_distinct_values(group[field]) if field in group.columns else ""
        summary_rows.append(summary_row)

    if not summary_rows:
        return pd.DataFrame(columns=output_columns)

    return (
        pd.DataFrame(summary_rows)
        .reindex(columns=output_columns)
        .sort_values(["document_count_sum", "review_row_count", group_column], ascending=[False, False, True])
        .reset_index(drop=True)
    )


def prepare_review_frame_for_excel(frame):
    excel_frame = frame.copy()
    for column in excel_frame.columns:
        if not (pd.api.types.is_object_dtype(excel_frame[column]) or pd.api.types.is_string_dtype(excel_frame[column])):
            continue
        non_null_index = excel_frame.index[excel_frame[column].notna()]
        if len(non_null_index) == 0:
            continue
        text_values = excel_frame.loc[non_null_index, column].astype(str)
        text_values = text_values.str.replace(REVIEW_EXCEL_ILLEGAL_CHARACTERS_RE, " ", regex=True)
        too_long_index = text_values[text_values.str.len().gt(REVIEW_EXCEL_CELL_CHAR_LIMIT)].index
        if len(too_long_index):
            text_values.loc[too_long_index] = text_values.loc[too_long_index].str.slice(0, REVIEW_EXCEL_CELL_CHAR_LIMIT)
        excel_frame.loc[non_null_index, column] = text_values
    return excel_frame


def write_review_conflict_workbook_to_path(conflict_frames, output_file):
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for sheet_name, frame in conflict_frames.items():
            prepare_review_frame_for_excel(frame).to_excel(writer, sheet_name=sheet_name, index=False)

        workbook = writer.book
        for worksheet in workbook.worksheets:
            worksheet.freeze_panes = "A2"
            worksheet.auto_filter.ref = worksheet.dimensions
            for column_cells in worksheet.columns:
                header = str(column_cells[0].value or "")
                column_letter = column_cells[0].column_letter
                max_len = len(header)
                for cell in column_cells[1: min(len(column_cells), 200)]:
                    value = "" if cell.value is None else str(cell.value)
                    max_len = max(max_len, min(len(value), 80))
                worksheet.column_dimensions[column_letter].width = min(max(max_len + 2, 10), 65)


def write_review_conflict_workbook(conflict_frames):
    try:
        write_review_conflict_workbook_to_path(conflict_frames, review_conflict_workbook_file)
        return review_conflict_workbook_file
    except PermissionError:
        fallback_file = review_conflict_workbook_file.with_name(
            f"{review_conflict_workbook_file.stem}_new{review_conflict_workbook_file.suffix}"
        )
        print(
            f"Could not overwrite {review_conflict_workbook_file}. "
            f"It may be open in Excel. Writing the checkpoint workbook to {fallback_file} instead."
        )
        write_review_conflict_workbook_to_path(conflict_frames, fallback_file)
        return fallback_file


def export_review_conflict_checkpoint(review):
    # Build conflict checks after accept/relabel/fallback logic, but before raw_lookup_key deduplication.
    retained_review_rows = review.loc[
        review["is_retained_manual_decision"] & review["raw_lookup_key"].ne("")
    ].copy()

    final_name_conflicts = summarize_review_conflicts(
        retained_review_rows,
        group_column="standardized_name",
        conflict_fields=REVIEW_METADATA_CONFLICT_FIELDS,
    )
    raw_lookup_conflicts = summarize_review_conflicts(
        retained_review_rows,
        group_column="raw_lookup_key",
        conflict_fields=RAW_LOOKUP_CONFLICT_FIELDS,
    )

    conflict_frames = {
        "final_name_conflict_summary": final_name_conflicts,
        "raw_key_conflict_summary": raw_lookup_conflicts,
    }
    written_conflict_workbook_file = write_review_conflict_workbook(conflict_frames)

    conflict_counts = {
        "final_name_conflicts": len(final_name_conflicts),
        "raw_lookup_key_conflicts": len(raw_lookup_conflicts),
        "workbook_file": written_conflict_workbook_file,
    }

    print(f"Wrote review conflict workbook: {written_conflict_workbook_file}")
    print(
        "Review conflicts found: "
        f"{conflict_counts['final_name_conflicts']:,} final standardized names; "
        f"{conflict_counts['raw_lookup_key_conflicts']:,} raw lookup keys."
    )

    return retained_review_rows, final_name_conflicts, raw_lookup_conflicts, conflict_counts


def load_manual_review_decisions():
    review = read_review_workbook(manual_review_file).fillna("")

    require_columns(
        review,
        {
            "raw_name",
            "document_count",
            "proposed_standardized_name_document_count",
            "review_action",
            "proposed_standardized_name",
            "proposed_entity_type",
            "proposed_data_holder",
            "reviewed_standardized_name",
            "reviewed_entity_type",
            "reviewed_data_holder",
        },
        "Manual review workbook",
    )

    for column in ["reviewer", "review_note", "data_holder_type", "URL"]:
        ensure_column(review, column)
    ensure_column(review, "data_holder_country_ISO2", aliases=["data_holder_country_iso2", "data_holder_country"])

    review = review.copy()
    review["raw_name"] = review["raw_name"].map(clean_text_value)
    review["raw_lookup_key"] = review["raw_name"].map(review_lookup_key)
    review["document_count"] = pd.to_numeric(review["document_count"], errors="coerce").fillna(0)
    review["proposed_standardized_name_document_count"] = pd.to_numeric(
        review["proposed_standardized_name_document_count"], errors="coerce"
    ).fillna(0)
    review["review_action_normalized"] = review["review_action"].map(normalize_review_action)

    accept_mask = review["review_action_normalized"].eq("accept")
    relabel_mask = review["review_action_normalized"].eq("relabel")

    proposed_name = review["proposed_standardized_name"].map(clean_text_value)
    reviewed_name = review["reviewed_standardized_name"].map(clean_text_value)
    proposed_entity_type = review["proposed_entity_type"].map(normalize_entity_type)
    reviewed_entity_type = review["reviewed_entity_type"].map(normalize_entity_type)
    proposed_data_holder = review["proposed_data_holder"].map(clean_text_value)
    reviewed_data_holder = review["reviewed_data_holder"].map(clean_text_value)

    review["standardized_name"] = ""
    review.loc[accept_mask, "standardized_name"] = proposed_name.loc[accept_mask]
    review.loc[relabel_mask, "standardized_name"] = reviewed_name.loc[relabel_mask].where(
        reviewed_name.loc[relabel_mask].ne(""),
        proposed_name.loc[relabel_mask],
    )

    review["entity_type"] = ""
    review.loc[accept_mask, "entity_type"] = proposed_entity_type.loc[accept_mask]
    review.loc[relabel_mask, "entity_type"] = reviewed_entity_type.loc[relabel_mask].where(
        reviewed_entity_type.loc[relabel_mask].ne(""),
        proposed_entity_type.loc[relabel_mask],
    )

    review["data_holder"] = ""
    review.loc[accept_mask, "data_holder"] = proposed_data_holder.loc[accept_mask]
    review.loc[relabel_mask, "data_holder"] = reviewed_data_holder.loc[relabel_mask].where(
        reviewed_data_holder.loc[relabel_mask].ne(""),
        proposed_data_holder.loc[relabel_mask],
    )

    review["relabel_fell_back_to_proposed_standardized_name"] = relabel_mask & reviewed_name.eq("") & proposed_name.ne("")
    review["relabel_fell_back_to_proposed_entity_type"] = relabel_mask & reviewed_entity_type.eq("") & proposed_entity_type.ne("")
    review["relabel_fell_back_to_proposed_data_holder"] = relabel_mask & reviewed_data_holder.eq("") & proposed_data_holder.ne("")

    for column in ["data_holder_type", "data_holder_country_ISO2", "URL"]:
        review[column] = review[column].map(clean_text_value)

    usable_action = review["review_action_normalized"].isin(REVIEW_ACTIONS_TO_USE)
    repeated_raw_name = review["document_count"].ge(MIN_REVIEW_DOCUMENT_COUNT)
    frequent_standardized_name = review["proposed_standardized_name_document_count"].gt(
        MIN_STANDARDIZED_NAME_DOCUMENT_COUNT_EXCLUSIVE
    )
    selected_entity_type = review["entity_type"].isin(INCLUDED_ENTITY_TYPES)

    invalid_selected_review_rows = review.loc[
        usable_action
        & repeated_raw_name
        & frequent_standardized_name
        & selected_entity_type
        & review["standardized_name"].eq(""),
        [
            "raw_name",
            "document_count",
            "proposed_standardized_name_document_count",
            "review_action",
            "proposed_standardized_name",
            "proposed_entity_type",
            "reviewed_standardized_name",
            "reviewed_entity_type",
        ],
    ].copy()
    if not invalid_selected_review_rows.empty:
        examples = invalid_selected_review_rows.head(20).to_dict("records")
        raise ValueError(
            "Some accepted/relabelled review rows with retained entity types, document_count >= "
            f"{MIN_REVIEW_DOCUMENT_COUNT}, and proposed standardized-name document count > "
            f"{MIN_STANDARDIZED_NAME_DOCUMENT_COUNT_EXCLUSIVE} are missing a final standardized name. "
            f"Examples: {examples}"
        )

    review["is_retained_manual_decision"] = (
        usable_action
        & repeated_raw_name
        & frequent_standardized_name
        & selected_entity_type
    )

    (
        retained_review_rows,
        final_name_conflicts,
        raw_lookup_conflicts,
        review_conflict_counts,
    ) = export_review_conflict_checkpoint(review)

    if STOP_FOR_MANUAL_CONFLICT_REVIEW and (
        review_conflict_counts["final_name_conflicts"] > 0
        or review_conflict_counts["raw_lookup_key_conflicts"] > 0
    ):
        raise RuntimeError(
            "Review metadata conflicts were found. Inspect and resolve "
            f"{review_conflict_counts['workbook_file']}, update {manual_review_file}, and rerun notebook 6. "
            "Set STOP_FOR_MANUAL_CONFLICT_REVIEW = False only if you deliberately want to continue despite conflicts."
        )

    review_decisions = (
        retained_review_rows
        .sort_values(["document_count", "standardized_name", "raw_name"], ascending=[False, True, True])
        .drop_duplicates("raw_lookup_key")
    )

    review_lookup = review_decisions[
        [
            "raw_lookup_key",
            "raw_name",
            "document_count",
            "review_action",
            "review_action_normalized",
            "standardized_name",
            "entity_type",
            "data_holder",
            "data_holder_type",
            "data_holder_country_ISO2",
            "URL",
            "reviewer",
            "review_note",
            "proposed_standardized_name",
            "proposed_standardized_name_document_count",
            "proposed_entity_type",
            "proposed_data_holder",
            "reviewed_standardized_name",
            "reviewed_entity_type",
            "reviewed_data_holder",
        ]
    ].rename(
        columns={
            "raw_name": "review_raw_name",
            "document_count": "review_document_count",
            "proposed_standardized_name_document_count": "review_proposed_standardized_name_document_count",
        }
    )

    action_counts = review["review_action_normalized"].replace("", "blank").value_counts(dropna=False)
    audit_rows = [
        {"metric": "manual_review_file", "value": str(manual_review_file)},
        {"metric": "manual_review_rows", "value": len(review)},
        {"metric": "manual_review_blank_action_rows", "value": int(review["review_action_normalized"].eq("").sum())},
        {"metric": "manual_review_rows_raw_document_count_ge_2", "value": int(repeated_raw_name.sum())},
        {"metric": "manual_review_rows_proposed_standardized_name_document_count_gt_50", "value": int(frequent_standardized_name.sum())},
        {"metric": "manual_review_rows_accepted_or_relabelled", "value": int(usable_action.sum())},
        {"metric": "manual_review_rows_retained_after_all_review_filters", "value": int(review["is_retained_manual_decision"].sum())},
        {"metric": "manual_review_unique_raw_names_retained", "value": review_decisions["raw_lookup_key"].nunique(dropna=True)},
        {"metric": "manual_review_final_standardized_name_conflicts", "value": review_conflict_counts["final_name_conflicts"]},
        {"metric": "manual_review_raw_lookup_key_conflicts", "value": review_conflict_counts["raw_lookup_key_conflicts"]},
        {"metric": "relabel_rows_fell_back_to_proposed_standardized_name", "value": int(review["relabel_fell_back_to_proposed_standardized_name"].sum())},
        {"metric": "relabel_rows_fell_back_to_proposed_entity_type", "value": int(review["relabel_fell_back_to_proposed_entity_type"].sum())},
        {"metric": "relabel_rows_fell_back_to_proposed_data_holder", "value": int(review["relabel_fell_back_to_proposed_data_holder"].sum())},
    ]
    audit_rows.extend(
        {"metric": f"manual_review_action_rows_{action}", "value": int(count)}
        for action, count in action_counts.items()
    )

    return review_lookup, pd.DataFrame(audit_rows)


def load_document_level_source_mentions(target_raw_lookup_keys):
    master_columns = pd.read_csv(llm_master_file, nrows=0).columns.tolist()
    require_columns(
        pd.DataFrame(columns=master_columns),
        {"db_use", "data_sources_json", "file_name"},
        "LLM master file",
    )

    desired_columns = [
        "global_index",
        "file_name",
        "relative_path",
        "source_subfolder",
        "db_use",
        "data_sources_json",
        "reason_short",
        "run_type",
        "run_name",
        "prompt_revision",
        "status",
        "is_success",
    ]
    usecols = [column for column in desired_columns if column in master_columns]
    target_raw_lookup_keys = set(target_raw_lookup_keys)

    source_records = []
    rows_scanned = 0
    successful_rows = 0
    data_use_rows = 0
    parsed_source_mentions = 0
    retained_source_mentions = 0

    for chunk_number, chunk in enumerate(pd.read_csv(llm_master_file, usecols=usecols, chunksize=chunk_size, low_memory=False), start=1):
        rows_scanned += len(chunk)

        if "is_success" in chunk.columns:
            is_success = chunk["is_success"]
            if is_success.dtype == object:
                is_success = is_success.astype("string").str.lower().isin({"true", "1", "yes"})
            chunk = chunk.loc[is_success.fillna(False)].copy()
        successful_rows += len(chunk)

        db_use_numeric = pd.to_numeric(chunk["db_use"], errors="coerce")
        chunk = chunk.loc[db_use_numeric.eq(1)].copy()
        data_use_rows += len(chunk)

        if chunk.empty:
            continue

        for row in chunk.itertuples(index=False):
            row_values = row._asdict()
            sources = parse_data_sources(row_values.get("data_sources_json"))
            if not sources:
                continue

            base = {
                "global_index": row_values.get("global_index"),
                "relative_path": row_values.get("relative_path"),
                "source_subfolder": row_values.get("source_subfolder"),
                "file_name": row_values.get("file_name"),
                "doi": source_doi_from_file_name(row_values.get("file_name")),
                "db_use": row_values.get("db_use"),
                "reason_short": row_values.get("reason_short"),
                "run_type": row_values.get("run_type"),
                "run_name": row_values.get("run_name"),
                "prompt_revision": row_values.get("prompt_revision"),
                "status": row_values.get("status"),
            }

            for source_order, source_name in enumerate(sources, start=1):
                raw_name = clean_text_value(source_name)
                if not raw_name:
                    continue
                parsed_source_mentions += 1
                raw_lookup_key = review_lookup_key(raw_name)
                if raw_lookup_key not in target_raw_lookup_keys:
                    continue
                retained_source_mentions += 1
                source_records.append(
                    {
                        **base,
                        "source_order": source_order,
                        "raw_name": raw_name,
                        "raw_lookup_key": raw_lookup_key,
                    }
                )

        if chunk_number % 10 == 0:
            print(
                f"Scanned {rows_scanned:,} master rows; retained {retained_source_mentions:,} source mentions so far.",
                flush=True,
            )

    source_rows = pd.DataFrame(source_records)
    source_audit = pd.DataFrame(
        [
            {"metric": "llm_master_file", "value": str(llm_master_file)},
            {"metric": "llm_master_rows_scanned", "value": rows_scanned},
            {"metric": "llm_master_successful_rows", "value": successful_rows},
            {"metric": "llm_master_db_use_1_rows", "value": data_use_rows},
            {"metric": "llm_master_parsed_source_mentions", "value": parsed_source_mentions},
            {"metric": "llm_master_source_mentions_matching_retained_review_names", "value": retained_source_mentions},
        ]
    )
    return source_rows, source_audit


def build_standardized_source_mentions():
    review_lookup, review_audit = load_manual_review_decisions()
    long_source_rows, source_audit = load_document_level_source_mentions(review_lookup["raw_lookup_key"].dropna())

    standardized_rows = long_source_rows.merge(
        review_lookup,
        on="raw_lookup_key",
        how="inner",
    )
    standardized_rows["standardization_source"] = "manual_review_" + standardized_rows["review_action_normalized"]
    standardized_rows["is_included_for_bibliometric_merge"] = True
    standardized_rows = apply_final_supplement_labels(standardized_rows)

    priority_columns = [
        "global_index",
        "relative_path",
        "source_subfolder",
        "file_name",
        "doi",
        "db_use",
        "source_order",
        "raw_name",
        "standardized_name",
        "entity_type",
        "data_holder",
        "data_holder_for_visualization",
        "data_holder_type",
        "data_holder_country_ISO2",
        "URL",
        "review_document_count",
        "review_proposed_standardized_name_document_count",
        "review_action",
        "standardization_source",
        "reviewer",
        "review_note",
        "proposed_standardized_name",
        "proposed_entity_type",
        "proposed_data_holder",
        "reviewed_standardized_name",
        "reviewed_entity_type",
        "reviewed_data_holder",
        "reason_short",
        "run_type",
        "run_name",
        "prompt_revision",
        "status",
    ]
    standardized_rows = standardized_rows[
        [column for column in priority_columns if column in standardized_rows.columns]
        + [column for column in standardized_rows.columns if column not in priority_columns and column != "raw_lookup_key"]
    ]

    standardization_audit = pd.concat(
        [
            review_audit,
            source_audit,
            pd.DataFrame(
                [
                    {"metric": "standardized_source_mentions_retained", "value": len(standardized_rows)},
                    {"metric": "standardized_unique_files", "value": standardized_rows["file_name"].nunique(dropna=True)},
                    {"metric": "standardized_unique_raw_names", "value": standardized_rows["raw_name"].nunique(dropna=True)},
                    {"metric": "standardized_unique_names", "value": standardized_rows["standardized_name"].nunique(dropna=True)},
                ]
            ),
        ],
        ignore_index=True,
    )

    standardized_rows.to_csv(source_mentions_long_file, index=False, encoding="utf-8-sig")
    standardization_audit.to_csv(standardization_audit_file, index=False, encoding="utf-8-sig")

    return standardized_rows, standardization_audit


def build_category_aware_labels():
    source_rows, standardization_audit = build_standardized_source_mentions()

    required_columns = {"file_name", "raw_name", "standardized_name", "entity_type"}
    missing_columns = required_columns - set(source_rows.columns)
    if missing_columns:
        raise ValueError(
            "Standardized long table is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    source_rows = source_rows.copy()
    for column in [
        "raw_name",
        "standardized_name",
        "entity_type",
        "data_holder",
        "data_holder_for_visualization",
        "data_holder_type",
        "data_holder_country_ISO2",
        "URL",
    ]:
        if column not in source_rows.columns:
            source_rows[column] = ""
        source_rows[column] = source_rows[column].fillna("").astype(str).str.strip()

    included_source_rows = source_rows.loc[
        source_rows["is_included_for_bibliometric_merge"]
        & source_rows["standardized_name"].astype(str).str.strip().ne("")
    ].copy()

    group_columns = [column for column in TEXT_IDENTIFIER_COLUMNS if column in included_source_rows.columns]
    if included_source_rows.empty:
        category_labels = pd.DataFrame(
            columns=group_columns
            + [
                "databases",
                "raw_data_sources",
                "source_entity_types",
                "source_data_holders",
                "source_data_holder_types",
                "source_data_holder_country_ISO2",
                "source_urls",
            ]
        )
    else:
        category_labels = (
            included_source_rows
            .groupby(group_columns, dropna=False)
            .agg(
                databases=("standardized_name", ordered_unique_semicolon_join),
                raw_data_sources=("raw_name", ordered_unique_join),
                source_entity_types=("entity_type", ordered_unique_semicolon_join),
                source_data_holders=("data_holder", ordered_unique_semicolon_join),
                source_data_holder_types=("data_holder_type", ordered_unique_semicolon_join),
                source_data_holder_country_ISO2=("data_holder_country_ISO2", ordered_unique_semicolon_join),
                source_urls=("URL", ordered_unique_join),
            )
            .reset_index()
        )

    for entity_type in INCLUDED_ENTITY_TYPES:
        subset = included_source_rows.loc[included_source_rows["entity_type"].eq(entity_type)]
        column_name = category_column_name(entity_type)
        if subset.empty:
            category_labels[column_name] = ""
            continue
        category_frame = (
            subset
            .groupby(group_columns, dropna=False)["standardized_name"]
            .agg(ordered_unique_semicolon_join)
            .reset_index(name=column_name)
        )
        category_labels = category_labels.merge(category_frame, on=group_columns, how="left")
        category_labels[column_name] = category_labels[column_name].fillna("")

    category_labels["file_name_key"] = category_labels["file_name"].map(normalize_file_name_key)

    entity_summary = (
        source_rows.groupby(["entity_type", "is_included_for_bibliometric_merge"], dropna=False)
        .size()
        .reset_index(name="source_mentions")
        .sort_values(["is_included_for_bibliometric_merge", "source_mentions"], ascending=[False, False])
    )

    return category_labels, included_source_rows, entity_summary, standardization_audit


labels, included_category_source_rows, entity_type_merge_summary, standardization_audit = build_category_aware_labels()

invalid_label_rows = labels[labels["file_name_key"].isna()].copy()
duplicate_label_file_names = (
    labels["file_name_key"]
    .value_counts(dropna=True)
    .rename_axis("file_name_key")
    .reset_index(name="count")
    .query("count > 1")
)

print(
    f"Built {len(labels):,} category-aware source-label rows from manual review decisions "
    f"and {llm_master_file.name}."
)
print(f"Wrote standardized source mentions to {source_mentions_long_file.name}: {len(included_category_source_rows):,}")
print("\nStandardization audit:")
print(standardization_audit.to_string(index=False))
print("\nEntity-type merge summary:")
print(entity_type_merge_summary.to_string(index=False))

assert invalid_label_rows.empty, "Some category-aware label rows do not have a usable file_name."
assert duplicate_label_file_names.empty, "Some category-aware label rows collapse to duplicate file_name keys."

labels.head()


In [ ]:
target_file_name_keys = set(labels["file_name_key"].dropna())
matched_chunks = []
rows_scanned = 0

for chunk_number, chunk in enumerate(pd.read_csv(merged_all_file, chunksize=chunk_size), start=1):
    rows_scanned += len(chunk)
    chunk["doi_normalized"] = chunk["doi"].map(normalize_doi)
    chunk["file_name_key"] = chunk["doi"].map(doi_to_file_name_key)
    matched_chunk = chunk[chunk["file_name_key"].isin(target_file_name_keys)].copy()

    if not matched_chunk.empty:
        matched_chunks.append(matched_chunk)
        print(f"Chunk {chunk_number}: matched {len(matched_chunk):,} bibliometric rows.")

bibliometric_matches = (
    pd.concat(matched_chunks, ignore_index=True)
    if matched_chunks
    else pd.DataFrame(columns=["file_name_key", "doi_normalized"])
)

duplicate_bibliometric_file_names = (
    bibliometric_matches["file_name_key"]
    .value_counts(dropna=True)
    .rename_axis("file_name_key")
    .reset_index(name="count")
    .query("count > 1")
)

assert duplicate_bibliometric_file_names.empty, (
    "Some standardized label file-name keys map to multiple rows in merged_all.csv. "
    "Inspect duplicate_bibliometric_file_names before merging."
)

bibliometric_matches = bibliometric_matches.rename(columns={"doi": "doi_bibliometric"})

print(f"\nScanned {rows_scanned:,} bibliometric rows.")
print(f"Matched {len(bibliometric_matches):,} bibliometric rows.")
print(
    f"Matched unique file-name keys: {bibliometric_matches['file_name_key'].nunique(dropna=True):,}"
)

preview_columns = [
    column
    for column in ["file_name_key", "doi_bibliometric", "title", "affiliations_countries"]
    if column in bibliometric_matches.columns
]
bibliometric_matches[preview_columns].head()


In [ ]:
bibliometric_matches.head()

In [ ]:
from collections import Counter
Counter(bibliometric_matches['asjc_subjects']).most_common()

In [ ]:
merged_output = labels.merge(
    bibliometric_matches,
    on="file_name_key",
    how="left",
)

merged_output["merge_status"] = merged_output["doi_bibliometric"].notna().map(
    {True: "matched", False: "missing_in_merged_all"}
)

if "Unnamed: 0" in merged_output.columns:
    merged_output = merged_output.drop(columns=["Unnamed: 0"])

unmatched_labels = merged_output.loc[
    merged_output["merge_status"].eq("missing_in_merged_all"),
    [
        column
        for column in [
            "global_index",
            "file_name",
            "doi",
            "relative_path",
            "source_subfolder",
            "file_name_key",
            "databases",
            "source_entity_types",
            "raw_data_sources",
            "data_resource_or_product_names",
            "data_holder_names",
            "source_data_holders",
            "source_data_holder_types",
            "source_data_holder_country_ISO2",
        ]
        if column in merged_output.columns
    ],
].copy()

priority_columns = [
    "global_index",
    "file_name",
    "doi",
    "relative_path",
    "source_subfolder",
    "file_name_key",
    "doi_bibliometric",
    "doi_normalized",
    "merge_status",
    "databases",
    "source_entity_types",
    "raw_data_sources",
    "data_resource_or_product_names",
    "data_holder_names",
    "source_data_holders",
    "source_data_holder_types",
    "source_data_holder_country_ISO2",
    "title",
    "asjc_subjects",
    "fields_of_study",
    "affiliations",
    "affiliations_countries",
]
priority_columns = [column for column in priority_columns if column in merged_output.columns]

ordered_columns = priority_columns + [
    column for column in merged_output.columns if column not in priority_columns
]
merged_output = merged_output[ordered_columns]

sort_columns = [
    column
    for column in ["merge_status", "source_subfolder", "global_index", "file_name"]
    if column in merged_output.columns
]
merged_output = merged_output.sort_values(sort_columns).reset_index(drop=True)

asjc_export_columns = [
    "global_index",
    "file_name",
    "doi",
    "doi_bibliometric",
    "merge_status",
    "title",
    "asjc_subjects",
    "fields_of_study",
    "databases",
    "source_entity_types",
    "data_resource_or_product_names",
    "data_holder_names",
    "source_data_holders",
    "source_data_holder_types",
    "source_data_holder_country_ISO2",
]
asjc_subject_fields = merged_output.loc[
    merged_output["merge_status"].eq("matched"),
    [column for column in asjc_export_columns if column in merged_output.columns],
].copy()

if "asjc_subjects" in asjc_subject_fields.columns:
    asjc_subject_fields["asjc_subjects_joined"] = asjc_subject_fields["asjc_subjects"].map(join_list_field)
if "fields_of_study" in asjc_subject_fields.columns:
    asjc_subject_fields["fields_of_study_joined"] = asjc_subject_fields["fields_of_study"].map(join_list_field)

asjc_subject_rows = []
if "asjc_subjects" in asjc_subject_fields.columns:
    for row in asjc_subject_fields.to_dict("records"):
        for subject_order, subject in enumerate(parse_list_field(row.get("asjc_subjects")), start=1):
            asjc_subject_rows.append(
                {
                    **row,
                    "asjc_subject_order": subject_order,
                    "asjc_subject": subject,
                }
            )

asjc_subjects_long = pd.DataFrame(asjc_subject_rows)
asjc_long_priority_columns = [
    "global_index",
    "file_name",
    "doi",
    "doi_bibliometric",
    "title",
    "asjc_subject_order",
    "asjc_subject",
    "asjc_subjects_joined",
    "fields_of_study_joined",
    "databases",
    "source_entity_types",
    "data_resource_or_product_names",
    "data_holder_names",
    "source_data_holders",
    "source_data_holder_types",
    "source_data_holder_country_ISO2",
]
if not asjc_subjects_long.empty:
    asjc_subjects_long = asjc_subjects_long[
        [column for column in asjc_long_priority_columns if column in asjc_subjects_long.columns]
        + [column for column in asjc_subjects_long.columns if column not in asjc_long_priority_columns]
    ]

matched_rows_with_asjc_subjects = (
    asjc_subject_fields["asjc_subjects_joined"].astype(str).str.strip().ne("").sum()
    if "asjc_subjects_joined" in asjc_subject_fields.columns
    else 0
)
matched_rows_with_fields_of_study = (
    asjc_subject_fields["fields_of_study_joined"].astype(str).str.strip().ne("").sum()
    if "fields_of_study_joined" in asjc_subject_fields.columns
    else 0
)

coverage_summary = pd.DataFrame(
    {
        "metric": [
            "standardized_label_rows",
            "unique_label_file_names",
            "matched_rows",
            "matched_unique_file_names",
            "unmatched_rows",
            "unmatched_unique_file_names",
        ],
        "value": [
            len(labels),
            labels["file_name_key"].nunique(dropna=True),
            merged_output["merge_status"].eq("matched").sum(),
            merged_output.loc[
                merged_output["merge_status"].eq("matched"), "file_name_key"
            ].nunique(dropna=True),
            len(unmatched_labels),
            unmatched_labels["file_name_key"].nunique(dropna=True),
        ],
    }
)

category_metric_rows = [
    {"metric": "included_source_mentions", "value": len(included_category_source_rows)},
    {
        "metric": "included_label_rows_with_any_category",
        "value": labels["databases"].astype(str).str.strip().ne("").sum(),
    },
]
for row in entity_type_merge_summary.itertuples(index=False):
    inclusion = "included" if row.is_included_for_bibliometric_merge else "excluded"
    category_metric_rows.append(
        {
            "metric": f"source_mentions_{inclusion}_{row.entity_type}",
            "value": int(row.source_mentions),
        }
    )

asjc_metric_rows = [
    {"metric": "matched_rows_with_asjc_subjects", "value": int(matched_rows_with_asjc_subjects)},
    {"metric": "matched_rows_with_fields_of_study", "value": int(matched_rows_with_fields_of_study)},
    {"metric": "asjc_subject_rows", "value": len(asjc_subjects_long)},
]

coverage_summary = pd.concat(
    [standardization_audit, coverage_summary, pd.DataFrame(category_metric_rows + asjc_metric_rows)],
    ignore_index=True,
)

print(coverage_summary.to_string(index=False))

if unmatched_labels.empty:
    print("\nAll standardized source-label rows were found in merged_all.csv.")
else:
    print("\nStandardized source-label rows missing from merged_all.csv:")
    print(unmatched_labels.head(30).to_string(index=False))

merged_output.head()


In [ ]:
def parse_publication_year(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (int, np.integer)):
        year = int(value)
        return year if 1800 <= year <= 2100 else pd.NA
    if isinstance(value, (float, np.floating)):
        if not np.isfinite(value):
            return pd.NA
        year = int(value)
        return year if 1800 <= year <= 2100 else pd.NA
    text = str(value).strip()
    if not text or text.lower() in {"nan", "<na>", "none"}:
        return pd.NA
    match = re.search(r"(18|19|20)\d{2}", text)
    if not match:
        return pd.NA
    year = int(match.group(0))
    return year if 1800 <= year <= 2100 else pd.NA


def unique_join_nonempty(values):
    return ordered_unique_join(values)


def parse_sources_for_supplement(value):
    return ordered_unique_join(parse_data_sources(value))


# Build the paper-level base for the supplementary workbook from the full LLM
# master file. This keeps all publications in sheet 1 for transparency, while
# raw extracted source names are shown only for rows where the LLM marked db_use=1.
def build_supplement_master_paper_rows():
    master_columns = pd.read_csv(llm_master_file, nrows=0).columns.tolist()
    required_columns = {"file_name", "db_use", "data_sources_json"}
    missing_columns = required_columns - set(master_columns)
    if missing_columns:
        raise ValueError(f"LLM master file is missing required columns: {sorted(missing_columns)}")

    use_columns = [
        column
        for column in ["global_index", "file_name", "doi", "db_use", "data_sources_json", "is_success"]
        if column in master_columns
    ]
    master_chunks = []
    row_offset = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(llm_master_file, usecols=use_columns, chunksize=chunk_size, low_memory=False),
        start=1,
    ):
        chunk = chunk.copy()
        chunk["supplement_row_order"] = np.arange(row_offset, row_offset + len(chunk))
        row_offset += len(chunk)

        chunk["file_name_key"] = chunk["file_name"].map(normalize_file_name_key)
        doi_from_file_name = chunk["file_name"].map(source_doi_from_file_name)
        if "doi" in chunk.columns:
            doi_from_master = chunk["doi"].map(clean_text_value)
            chunk["doi"] = doi_from_file_name.where(doi_from_file_name.ne(""), doi_from_master)
        else:
            chunk["doi"] = doi_from_file_name

        db_use_numeric = pd.to_numeric(chunk["db_use"], errors="coerce")
        chunk["raw_extracted_source_names"] = ""
        source_mask = db_use_numeric.eq(1)
        if source_mask.any():
            chunk.loc[source_mask, "raw_extracted_source_names"] = (
                chunk.loc[source_mask, "data_sources_json"].map(parse_sources_for_supplement)
            )

        master_chunks.append(
            chunk[
                [
                    "supplement_row_order",
                    "doi",
                    "file_name_key",
                    "raw_extracted_source_names",
                ]
            ]
        )

        if chunk_number % 5 == 0:
            print(f"Read {row_offset:,} LLM master rows for the supplementary paper sheet.")

    return pd.concat(master_chunks, ignore_index=True)


# Add bibliographic metadata from merged_all.csv. The join uses the same DOI-derived
# file_name_key as the source-label reconstruction above, so papers without retained
# source names can still appear in the supplementary paper sheet.
def load_bibliographic_metadata_for_file_keys(file_name_keys):
    merged_all_columns = pd.read_csv(merged_all_file, nrows=0).columns.tolist()
    if "doi" not in merged_all_columns:
        raise ValueError("merged_all.csv must contain a doi column to join bibliographic metadata.")

    metadata_columns = [
        column
        for column in ["doi", "title", "dates", "affiliations", "affiliations_countries", "asjc_subjects"]
        if column in merged_all_columns
    ]
    metadata_chunks = []
    rows_scanned_for_supplement = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(merged_all_file, usecols=metadata_columns, chunksize=chunk_size, low_memory=False),
        start=1,
    ):
        rows_scanned_for_supplement += len(chunk)
        chunk = chunk.copy()
        chunk["file_name_key"] = chunk["doi"].map(doi_to_file_name_key)
        matched_chunk = chunk[chunk["file_name_key"].isin(file_name_keys)].copy()
        if not matched_chunk.empty:
            metadata_chunks.append(matched_chunk)
            print(
                f"Supplement metadata chunk {chunk_number}: matched {len(matched_chunk):,} bibliometric rows."
            )

    if not metadata_chunks:
        empty_columns = [
            "file_name_key",
            "publication_year",
            "title",
            "affiliations",
            "affiliations_countries",
            "asjc_subjects",
        ]
        print(f"Scanned {rows_scanned_for_supplement:,} bibliometric rows for the supplementary paper sheet.")
        return pd.DataFrame(columns=empty_columns)

    metadata = pd.concat(metadata_chunks, ignore_index=True)
    duplicate_metadata_rows = metadata["file_name_key"].duplicated(keep="first").sum()
    if duplicate_metadata_rows:
        print(
            "Dropping "
            f"{duplicate_metadata_rows:,} duplicate bibliometric rows for supplementary metadata; "
            "keeping the first row per file-name key."
        )
    metadata = metadata.drop_duplicates("file_name_key", keep="first").copy()

    metadata["publication_year"] = (
        metadata["dates"].map(parse_publication_year).astype("Int64")
        if "dates" in metadata.columns
        else pd.Series(pd.array([pd.NA] * len(metadata), dtype="Int64"), index=metadata.index)
    )

    for column in ["title", "affiliations", "affiliations_countries", "asjc_subjects"]:
        if column not in metadata.columns:
            metadata[column] = ""

    print(f"Scanned {rows_scanned_for_supplement:,} bibliometric rows for the supplementary paper sheet.")
    print(f"Matched metadata for {metadata['file_name_key'].nunique(dropna=True):,} LLM master papers.")

    return metadata[
        [
            "file_name_key",
            "publication_year",
            "title",
            "affiliations",
            "affiliations_countries",
            "asjc_subjects",
        ]
    ]


supplement_master_papers = build_supplement_master_paper_rows()
master_file_name_keys = set(supplement_master_papers["file_name_key"].dropna().astype(str))
supplement_bibliographic_metadata = load_bibliographic_metadata_for_file_keys(master_file_name_keys)

# Retrospective fixes based on review of the preliminary supplementary workbook.
# This section is deliberately placed after the manual-review decisions are loaded
# and before the supplementary workbook is rebuilt. That makes the final exported
# workbook reproducible: rerunning notebook 6 reapplies these explicit corrections.
# Add narrow, named overrides here only after confirming that the reviewed metadata
# should be changed without reopening the full manual-review workbook.
# Use `replacement_standardized_name` only when two standardized source names should
# be merged into one final name; leave it blank for ordinary metadata corrections.
RETROSPECTIVE_STANDARDIZED_SOURCE_OVERRIDES = pd.DataFrame(
    [
        {
            "standardized_name": "UniProt",
            "data_holder": "",
            "data_holder_type": "",
            "data_holder_country_ISO2": "GB; CH; US",
            "URL": "",
            "reason": "Use ISO2 country code GB instead of UK.",
        },
        {
            "standardized_name": "Swiss-Prot",
            "data_holder": "",
            "data_holder_type": "",
            "data_holder_country_ISO2": "GB; CH; US",
            "URL": "",
            "reason": "Use ISO2 country code GB instead of UK.",
        },
        {
            "standardized_name": "UniRef",
            "data_holder": "",
            "data_holder_type": "",
            "data_holder_country_ISO2": "GB; CH; US",
            "URL": "",
            "reason": "Use ISO2 country code GB instead of UK.",
        },
        {
            "standardized_name": "TrEMBL",
            "data_holder": "",
            "data_holder_type": "",
            "data_holder_country_ISO2": "GB; CH; US",
            "URL": "",
            "reason": "Use ISO2 country code GB instead of UK.",
        },
        {
            "standardized_name": "ProteomeXchange",
            "data_holder": "",
            "data_holder_type": "",
            "data_holder_country_ISO2": "US; JP; GB; CH",
            "URL": "",
            "reason": "Use ISO2 country code GB instead of UK.",
        },
        {
            "standardized_name": "UNFCCC",
            "data_holder": "UNFCCC",
            "data_holder_type": "",
            "data_holder_country_ISO2": "",
            "URL": "",
            "reason": "Correct typo in data-holder name.",
        },
        {
            "standardized_name": "Finnish Meteorological Institute",
            "data_holder": "Finnish Meteorological Institute",
            "data_holder_type": "",
            "data_holder_country_ISO2": "",
            "URL": "",
            "reason": "Correct typo in data-holder name.",
        },
        {
            "standardized_name": "Catalogue of Life",
            "data_holder": "",
            "data_holder_type": "",
            "data_holder_country_ISO2": "",
            "URL": "https://www.catalogueoflife.org/",
            "reason": "Replace incorrect URL with Catalogue of Life homepage.",
        },
        {
            "standardized_name": "EzTaxon-e",
            "replacement_standardized_name": "EzTaxon",
            "data_holder": "EzBioCloud",
            "data_holder_type": "university",
            "data_holder_country_ISO2": "KR",
            "URL": "https://www.ezbiocloud.net/",
            "reason": "Merge EzTaxon-e into EzTaxon and keep the shared EzBioCloud metadata.",
        },
    ]
)


def apply_retrospective_source_metadata_overrides(source_mentions, overrides):
    updated = source_mentions.copy()
    applied_rows = []

    if overrides.empty:
        return updated, pd.DataFrame(columns=["standardized_name", "replacement_standardized_name", "rows_changed", "fields_changed", "reason"])

    for override in overrides.fillna("").to_dict("records"):
        standardized_name = clean_text_value(override.get("standardized_name", ""))
        if not standardized_name:
            continue

        mask = updated["standardized_name"].map(clean_text_value).eq(standardized_name)
        rows_changed = int(mask.sum())
        fields_changed = []

        if rows_changed:
            replacement_standardized_name = clean_text_value(override.get("replacement_standardized_name", ""))
            if replacement_standardized_name:
                updated.loc[mask, "standardized_name"] = replacement_standardized_name
                fields_changed.append("standardized_name")

            for column in ["data_holder", "data_holder_type", "data_holder_country_ISO2", "URL"]:
                value = clean_text_value(override.get(column, ""))
                if value:
                    updated.loc[mask, column] = value
                    fields_changed.append(column)

        applied_rows.append(
            {
                "standardized_name": standardized_name,
                "replacement_standardized_name": clean_text_value(override.get("replacement_standardized_name", "")),
                "rows_changed": rows_changed,
                "fields_changed": "; ".join(fields_changed),
                "reason": clean_text_value(override.get("reason", "")),
            }
        )

    return updated, pd.DataFrame(applied_rows)


# Start from the retained, manually reviewed mention-level rows. Retrospective
# overrides are applied at this level so a renamed source updates every downstream
# object: paper-level labels, standardized-source counts, and source metadata.
supplement_source_mentions = included_category_source_rows.copy()
supplement_source_mentions["file_name_key"] = supplement_source_mentions["file_name"].map(normalize_file_name_key)
supplement_source_mentions = supplement_source_mentions[
    supplement_source_mentions["standardized_name"].astype(str).str.strip().ne("")
].copy()
supplement_source_mentions, retrospective_override_audit = apply_retrospective_source_metadata_overrides(
    supplement_source_mentions,
    RETROSPECTIVE_STANDARDIZED_SOURCE_OVERRIDES,
)
# Reapply final display labels after retrospective fixes, because some fixes
# intentionally set names/types that should flow into the supplement outputs.
supplement_source_mentions = apply_final_supplement_labels(supplement_source_mentions)

if not retrospective_override_audit.empty:
    print("Retrospective source-metadata overrides applied:")
    display(retrospective_override_audit)

# Sheet 1 reports the two analysis-facing categories directly: data resources/products
# and standardized data holders. Sheet 2 uses the same split in row form.
# Resource rows keep the standardized resource name; holder rows leave it blank.
# The standardized holder column always comes from the reviewed data_holder field.
supplement_source_lookup_rows = supplement_source_mentions.copy()
resource_mask = supplement_source_lookup_rows["entity_type"].eq("data_resource_or_product")

supplement_source_lookup_rows["standardized_data_resource_name"] = ""
supplement_source_lookup_rows.loc[resource_mask, "standardized_data_resource_name"] = supplement_source_lookup_rows.loc[
    resource_mask,
    "standardized_name",
]
supplement_source_lookup_rows["data_resource_for_visualization"] = supplement_source_lookup_rows[
    "standardized_data_resource_name"
]

supplement_source_lookup_rows["standardized_data_holder_name"] = supplement_source_lookup_rows["data_holder"]
supplement_source_lookup_rows["data_holder_for_visualization"] = supplement_source_lookup_rows[
    "standardized_data_holder_name"
].map(final_data_holder_name_list)

source_label_keys = supplement_source_lookup_rows[["file_name_key"]].drop_duplicates().copy()
resource_labels_for_supplement = (
    supplement_source_lookup_rows[
        supplement_source_lookup_rows["standardized_data_resource_name"].map(clean_text_value).ne("")
    ]
    .groupby("file_name_key", dropna=False)["standardized_data_resource_name"]
    .agg(ordered_unique_semicolon_join)
    .reset_index(name="standardized_data_resource_names")
)
holder_labels_for_supplement = (
    supplement_source_lookup_rows[
        supplement_source_lookup_rows["standardized_data_holder_name"].map(clean_text_value).ne("")
    ]
    .groupby("file_name_key", dropna=False)["standardized_data_holder_name"]
    .agg(ordered_unique_semicolon_join)
    .reset_index(name="standardized_data_holder_names")
)
labels_for_supplement = (
    source_label_keys
    .merge(resource_labels_for_supplement, on="file_name_key", how="left")
    .merge(holder_labels_for_supplement, on="file_name_key", how="left")
)
for column in [
    "standardized_data_resource_names",
    "standardized_data_holder_names",
]:
    labels_for_supplement[column] = labels_for_supplement[column].fillna("")

supplement_paper_data = supplement_master_papers.merge(
    supplement_bibliographic_metadata,
    on="file_name_key",
    how="left",
).merge(
    labels_for_supplement,
    on="file_name_key",
    how="left",
)

for column in [
    "standardized_data_resource_names",
    "standardized_data_holder_names",
]:
    supplement_paper_data[column] = supplement_paper_data[column].fillna("").map(clean_text_value)
supplement_paper_data["raw_extracted_source_names"] = (
    supplement_paper_data["raw_extracted_source_names"].fillna("").map(clean_text_value)
)
supplement_paper_data["used_in_analysis"] = (
    supplement_paper_data["standardized_data_resource_names"].str.strip().ne("")
    | supplement_paper_data["standardized_data_holder_names"].str.strip().ne("")
)

for column in ["title", "affiliations", "affiliations_countries", "asjc_subjects"]:
    supplement_paper_data[column] = supplement_paper_data[column].fillna("")

supplement_paper_data = supplement_paper_data[
    [
        "supplement_row_order",
        "doi",
        "publication_year",
        "title",
        "affiliations",
        "affiliations_countries",
        "asjc_subjects",
        "raw_extracted_source_names",
        "standardized_data_resource_names",
        "standardized_data_holder_names",
        "used_in_analysis",
    ]
].sort_values(["publication_year", "doi", "supplement_row_order"], na_position="last")
supplement_paper_data = supplement_paper_data.drop(columns=["supplement_row_order"]).reset_index(drop=True)

# Sheet 2 is a raw-name lookup with separate canonical columns for resources/products and holders.
# It intentionally has no visualization-name columns; those are isolated in sheet 3 below.
supplement_standardized_sources = (
    supplement_source_lookup_rows
    .groupby(
        [
            "raw_name",
            "standardized_data_resource_name",
            "standardized_data_holder_name",
        ],
        dropna=False,
    )
    .agg(
        raw_name_document_count=("file_name_key", "nunique"),
        standardized_data_holder_type=("data_holder_type", ordered_unique_semicolon_join),
        standardized_data_holder_country_ISO2=("data_holder_country_ISO2", ordered_unique_semicolon_join),
        URL=("URL", unique_join_nonempty),
    )
    .reset_index()
    .sort_values(
        ["raw_name_document_count", "raw_name", "standardized_data_resource_name", "standardized_data_holder_name"],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)

# Sheet 3 is only for inspecting/editing paper-facing visualization names.
# It follows the same row logic as sheet 2: a row has both resource and holder
# fields when both exist, and only holder fields when the resource field is blank.
visualization_name_columns = [
    "standardized_data_resource_name",
    "data_resource_for_visualization",
    "standardized_data_holder_name",
    "data_holder_for_visualization",
    "standardized_data_holder_type",
    "standardized_data_holder_country_ISO2",
]

supplement_visualization_names = (
    supplement_source_lookup_rows
    .groupby(
        [
            "standardized_data_resource_name",
            "data_resource_for_visualization",
            "standardized_data_holder_name",
            "data_holder_for_visualization",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        standardized_data_holder_type=("data_holder_type", ordered_unique_semicolon_join),
        standardized_data_holder_country_ISO2=("data_holder_country_ISO2", ordered_unique_semicolon_join),
    )
    [visualization_name_columns]
    .sort_values(["standardized_data_resource_name", "standardized_data_holder_name"])
    .reset_index(drop=True)
)

print(f"Supplement paper rows: {len(supplement_paper_data):,}")
print(
    "Supplement paper rows with raw extracted names: "
    f"{supplement_paper_data['raw_extracted_source_names'].astype(str).str.strip().ne('').sum():,}"
)
print(f"Supplement paper rows used in analysis: {supplement_paper_data['used_in_analysis'].sum():,}")
print(
    "Supplement paper rows with bibliometric metadata: "
    f"{supplement_paper_data['title'].astype(str).str.strip().ne('').sum():,}"
)
print(f"Supplement standardized source rows: {len(supplement_standardized_sources):,}")
print(f"Supplement visualization-name rows: {len(supplement_visualization_names):,}")
print(f"Supplement workbook will be written to: {supplementary_workbook_file}")
supplement_paper_data.head()


In [ ]:
merged_output.to_csv(merged_output_file, index=False)
unmatched_labels.to_csv(unmatched_output_file, index=False)
coverage_summary.to_csv(coverage_output_file, index=False)
asjc_subject_fields.to_csv(asjc_subject_fields_output_file, index=False)
asjc_subjects_long.to_csv(asjc_subjects_long_output_file, index=False)

EXCEL_CELL_CHAR_LIMIT = 32_767
EXCEL_ILLEGAL_CHARACTERS_RE = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")


def prepare_for_excel(frame):
    excel_frame = frame.copy()
    truncated_cells = 0
    illegal_character_cells = 0

    for column in excel_frame.columns:
        if not (pd.api.types.is_object_dtype(excel_frame[column]) or pd.api.types.is_string_dtype(excel_frame[column])):
            continue

        non_null_index = excel_frame.index[excel_frame[column].notna()]
        if len(non_null_index) == 0:
            continue

        text_values = excel_frame.loc[non_null_index, column].astype(str)
        illegal_index = text_values[text_values.str.contains(EXCEL_ILLEGAL_CHARACTERS_RE, regex=True)].index
        illegal_character_cells += len(illegal_index)
        if len(illegal_index):
            text_values = text_values.str.replace(EXCEL_ILLEGAL_CHARACTERS_RE, " ", regex=True)

        too_long_index = text_values[text_values.str.len().gt(EXCEL_CELL_CHAR_LIMIT)].index
        truncated_cells += len(too_long_index)
        if len(too_long_index):
            text_values.loc[too_long_index] = text_values.loc[too_long_index].str.slice(0, EXCEL_CELL_CHAR_LIMIT)

        excel_frame.loc[non_null_index, column] = text_values

    return excel_frame, truncated_cells, illegal_character_cells


(
    supplement_paper_data_excel,
    paper_truncated_cells,
    paper_illegal_character_cells,
) = prepare_for_excel(supplement_paper_data)
(
    supplement_standardized_sources_excel,
    source_truncated_cells,
    source_illegal_character_cells,
) = prepare_for_excel(supplement_standardized_sources)
(
    supplement_visualization_names_excel,
    visualization_truncated_cells,
    visualization_illegal_character_cells,
) = prepare_for_excel(supplement_visualization_names)

with pd.ExcelWriter(supplementary_workbook_file, engine="openpyxl") as writer:
    supplement_paper_data_excel.to_excel(writer, sheet_name="paper_source_data", index=False)
    supplement_standardized_sources_excel.to_excel(writer, sheet_name="standardized_sources", index=False)
    supplement_visualization_names_excel.to_excel(writer, sheet_name="visualization_names", index=False)

    workbook = writer.book
    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        for column_cells in worksheet.columns:
            header = str(column_cells[0].value or "")
            column_letter = column_cells[0].column_letter
            max_len = len(header)
            for cell in column_cells[1: min(len(column_cells), 200)]:
                value = "" if cell.value is None else str(cell.value)
                max_len = max(max_len, min(len(value), 80))
            worksheet.column_dimensions[column_letter].width = min(max(max_len + 2, 10), 60)

print(f"Saved standardized source mentions to: {source_mentions_long_file}")
print(f"Saved standardization audit to: {standardization_audit_file}")
print(f"Saved merged output to: {merged_output_file}")
print(f"Saved unmatched report to: {unmatched_output_file}")
print(f"Saved coverage summary to: {coverage_output_file}")
print(f"Saved ASJC subject/field table to: {asjc_subject_fields_output_file}")
print(f"Saved ASJC subjects long table to: {asjc_subjects_long_output_file}")
print(f"Saved supplementary workbook to: {supplementary_workbook_file}")
print(f"Excel cell-length truncations in paper_source_data: {paper_truncated_cells}")
print(f"Excel cell-length truncations in standardized_sources: {source_truncated_cells}")
print(f"Excel cell-length truncations in visualization_names: {visualization_truncated_cells}")
print(f"Excel illegal-character cells cleaned in paper_source_data: {paper_illegal_character_cells}")
print(f"Excel illegal-character cells cleaned in standardized_sources: {source_illegal_character_cells}")
print(f"Excel illegal-character cells cleaned in visualization_names: {visualization_illegal_character_cells}")
